## 1 - imports and setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))

import config
from src.evaluation import (
    display_mod_performance,
    evaluate_performance,
    plot_gains_comparison,
    plot_pr_comparison,
    plot_roc_comparison,
)
from src.experiment import load_experiment_log
from src.mrm import get_gains, score_deciles

## 2 - load data and models

In [ ]:
# test data -- selected features for final models, full features for baseline/stage-6
X_test_sel = pd.read_parquet(config.RESULTS_PATH / "X_test_selected.parquet")
X_test     = pd.read_parquet(config.RESULTS_PATH / "X_test.parquet")
y_test     = pd.read_parquet(config.RESULTS_PATH / "y_test.parquet").squeeze()

models_dir = config.OUTPUT_PATH / 'models'

# baseline and full-feature models (trained on all features)
baseline_model = joblib.load(models_dir / "baseline.pkl")
cat_model      = joblib.load(models_dir / "catboost_model.pkl")
xgb_model      = joblib.load(models_dir / "xgboost_model.pkl")
lgb_booster    = joblib.load(models_dir / "lightgbm_model.pkl")

# final tuned models (trained on selected features)
cat_final   = joblib.load(models_dir / "catboost_final.pkl")
xgb_final   = joblib.load(models_dir / "xgboost_final.pkl")
lgb_final   = joblib.load(models_dir / "lightgbm_final.pkl")

# ensemble weights from experiment log val AUC
exp_log = load_experiment_log(config.RESULTS_PATH)
final_names = ['catboost_final', 'xgboost_final', 'lightgbm_final']
auc_vals = [
    exp_log[exp_log['model'] == name]['metrics.val_auc'].iloc[0]
    for name in final_names
]
raw_weights = np.array(auc_vals)
weights = raw_weights / raw_weights.sum()

print(f"X_test (full):     {X_test.shape}")
print(f"X_test (selected): {X_test_sel.shape}")
print(f"y_test fraud rate: {y_test.mean():.4f}")
print(f"models loaded: 7")
print(f"ensemble weights -- catboost: {weights[0]:.4f}  xgboost: {weights[1]:.4f}  lightgbm: {weights[2]:.4f}")

## 3 - generate test set predictions

In [ ]:
# full-feature models
baseline_proba  = baseline_model.predict_proba(X_test)[:, 1]
cat_proba       = cat_model.predict_proba(X_test)[:, 1]
xgb_proba       = xgb_model.predict_proba(X_test)[:, 1]
lgb_proba       = lgb_booster.predict(X_test)  # native booster

# final tuned models on selected features
cat_final_proba = cat_final.predict_proba(X_test_sel)[:, 1]
xgb_final_proba = xgb_final.predict_proba(X_test_sel)[:, 1]
lgb_final_proba = lgb_final.predict(X_test_sel)  # native booster

# weighted ensemble
ensemble_proba = (
    weights[0] * cat_final_proba
    + weights[1] * xgb_final_proba
    + weights[2] * lgb_final_proba
)

# master scores dataframe
df_scores = pd.DataFrame(
    {
        'baseline':         baseline_proba,
        'catboost':         cat_proba,
        'xgboost':          xgb_proba,
        'lightgbm':         lgb_proba,
        'catboost_final':   cat_final_proba,
        'xgboost_final':    xgb_final_proba,
        'lightgbm_final':   lgb_final_proba,
        'ensemble_weighted': ensemble_proba,
    },
    index=y_test.index,
)
print(f"scores dataframe: {df_scores.shape}")

## 4 - master comparison table

In [ ]:
comparison = pd.DataFrame([
    evaluate_performance(df_scores, col, y_test.values, threshold=0.8)
    for col in df_scores.columns
]).sort_values('AUC', ascending=False).reset_index(drop=True)

display(comparison)

**first use of test set** -- all prior model selection, feature selection, and hyperparameter tuning was performed on the validation set only. test AUC is an unbiased estimate of production performance.

**AUC improvement stages:**
- baseline (decision tree, all features): ~0.80
- stage 6 gradient boosted (all features): ~0.91-0.92
- final tuned (selected features): see table above
- ensemble (weighted avg of three finals): typically matches or exceeds best individual

## 5 - ROC curve comparison

In [ ]:
model_scores = dict(zip(df_scores.columns, df_scores.values.T))

config.PLOTS_PATH.mkdir(parents=True, exist_ok=True)
plot_roc_comparison(
    model_scores,
    y_test.values,
    output_path=config.PLOTS_PATH / "roc_comparison_final.png",
)

## 6 - gains curve comparison

In [ ]:
plot_gains_comparison(
    model_scores,
    y_test.values,
    output_path=config.PLOTS_PATH / "gains_comparison_final.png",
)

## 7 - PR curve comparison

In [ ]:
plot_pr_comparison(
    model_scores,
    y_test.values,
    output_path=config.PLOTS_PATH / "pr_curves_comparison_final.png",
)

## 8 - best model deep dive

In [ ]:
best_name  = comparison.iloc[0]['Score']
best_proba = df_scores[best_name].values
best_preds = (best_proba > 0.8).astype(int)
print(f"best model: {best_name}")

# confusion matrix
display_mod_performance(
    y_test.values, best_preds,
    save_path=str(config.PLOTS_PATH / "best_model_test_confusion.png"),
)

In [ ]:
# decile table
df_decile = pd.DataFrame({
    'TransactionID': range(len(y_test)),
    config.TARGET: y_test.values,
    best_name: best_proba,
})
decile_tbl = score_deciles(df_decile, uid='TransactionID', score=best_name, target=config.TARGET)
display(decile_tbl)

In [ ]:
get_gains(
    df_decile, target=config.TARGET, score=best_name, top_n=0.1,
    save_path=str(config.PLOTS_PATH / "best_model_test_gains.png"),
)

In [ ]:
# key stats at threshold 0.8
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
prec, rec, f1, _ = precision_recall_fscore_support(y_test.values, best_preds, average='binary')
auc  = roc_auc_score(y_test.values, best_proba)
top_decile_cap = decile_tbl.iloc[0]['fraud_pct_cap']

print(f"model:               {best_name}")
print(f"test AUC:            {auc:.5f}")
print(f"gini:                {2*auc-1:.5f}")
print(f"precision @ 0.8:     {prec:.5f}")
print(f"recall @ 0.8:        {rec:.5f}")
print(f"f1 @ 0.8:            {f1:.5f}")
print(f"fraud captured top decile: {top_decile_cap:.4f}")

## 9 - full experiment log

In [ ]:
exp_log_full = load_experiment_log(config.RESULTS_PATH)
display(exp_log_full)

In [ ]:
# AUC progression -- sort by timestamp ascending
exp_sorted = exp_log_full.sort_values('timestamp').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(range(len(exp_sorted)), exp_sorted['metrics.val_auc'], 'o-', color='steelblue', markersize=6)

for i, row in exp_sorted.iterrows():
    ax.annotate(
        row['model'],
        xy=(i, row['metrics.val_auc']),
        xytext=(0, 10),
        textcoords='offset points',
        ha='center', fontsize=7, rotation=40,
    )

ax.set_xticks(range(len(exp_sorted)))
ax.set_xticklabels(range(1, len(exp_sorted) + 1))
ax.set_xlabel('experiment run (chronological)')
ax.set_ylabel('val AUC')
ax.set_title('AUC Progression Across All Experiments')
fig.tight_layout()
fig.savefig(config.PLOTS_PATH / "auc_progression.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()

## 10 - final recommendation

**best model on test set** -- the ensemble or the best individual final model (see comparison table above). the ensemble typically matches or slightly exceeds the best single model by averaging out individual prediction errors.

**production recommendation** -- the weighted ensemble of catboost_final, xgboost_final, and lightgbm_final. rationale: ensemble reduces variance vs. any single model; all three constituent models were tuned on selected features reducing the input dimensionality and potential for overfitting; the AUC-based weighting ensures underperforming models contribute less.

**precision vs recall tradeoff** -- at threshold 0.8, the model is conservative: high precision (few false positives) but lower recall (some fraud missed). for a fraud team with limited review capacity, 0.8 is appropriate. if the cost of missed fraud is high relative to false positive cost, lower the threshold to 0.5-0.6 to increase recall at the expense of precision.

**suggested operating threshold** -- 0.5 for maximum fraud capture in low-stakes flagging workflows; 0.8 for high-confidence block decisions. use a tiered approach: block at 0.8+, review at 0.4-0.8, pass below 0.4.

**monitoring recommendations** --
- track score distribution weekly; shift in mean score indicates population drift
- monitor fraud rate in top decile; degradation signals model staleness
- retrain when test-period AUC drops more than 2 points below val AUC
- log all predictions with timestamps for post-hoc analysis
- review feature importance quarterly; new fraud patterns may require new features